# Course 5 - Matplotlib Essentials

Live-coding notebook that mirrors the slide deck.
Concrete examples distilled from Aurelien Geron's Matplotlib tutorial.

**Sections**
1. The Figure / Axes mental model (0:00-0:30)
2. The chart zoo (0:30-1:00)
3. Subplots & saving (1:00-1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shared.data_utils import load_dataset
rng = np.random.default_rng(0)


## 1. Figure -> Axes -> Artists

A **Figure** is the page. An **Axes** is one rectangular plot area on the page. Everything you draw - lines, markers, text - is an *Artist* on an Axes. `plt.subplots()` returns both, and that is the modern, explicit API.

In [ ]:
fig, ax = plt.subplots()
ax.plot([-3, -2, 5, 0], [1, 6, 4, 3]);


### Plot a function - labels, title, grid, legend

In [ ]:
x = np.linspace(-2, 2, 500)
fig, ax = plt.subplots()
ax.plot(x, x**2, label='y = x^2')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Square function')
ax.grid(True)
ax.legend();


### Line styles, colors, markers

In [ ]:
fig, ax = plt.subplots()
ax.plot(x, x,        label='x',     linestyle='-')
ax.plot(x, x**2,     label='x^2',   linestyle='--')
ax.plot(x, x**3,     label='x^3',   linestyle=':', marker='.', markevery=20)
ax.set_title('Powers of x')
ax.legend();


### Axis limits

In [ ]:
fig, ax = plt.subplots()
ax.plot(x, x**2)
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(0, 5)
ax.set_title('Zoomed');


## 2. The chart zoo

### Scatter - colored by a third variable

In [ ]:
penguins = load_dataset('penguins').dropna()
fig, ax = plt.subplots()
colors = penguins['species'].astype('category').cat.codes
ax.scatter(penguins['bill_length_mm'],
           penguins['bill_depth_mm'],
           c=colors, cmap='viridis', s=30, alpha=0.7)
ax.set_xlabel('bill length (mm)')
ax.set_ylabel('bill depth (mm)')
ax.set_title('Penguins - bill length vs depth');


### Bar

In [ ]:
counts = penguins['species'].value_counts()
fig, ax = plt.subplots()
ax.bar(counts.index, counts.values)
ax.set_ylabel('count')
ax.set_title('Penguins per species');


### Histogram

In [ ]:
fig, ax = plt.subplots()
ax.hist(penguins['body_mass_g'], bins=30, rwidth=0.9)
ax.set_xlabel('body mass (g)')
ax.set_ylabel('count');


### Boxplot

In [ ]:
groups = [penguins.loc[penguins['species'] == s, 'body_mass_g']
          for s in penguins['species'].unique()]
fig, ax = plt.subplots()
ax.boxplot(groups, tick_labels=list(penguins['species'].unique()))
ax.set_ylabel('body mass (g)');


## 3. Subplots & saving

### A 2x2 grid

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True)
x = np.linspace(-1.4, 1.4, 30)
axes[0, 0].plot(x, x);     axes[0, 0].set_title('x')
axes[0, 1].plot(x, x**2);  axes[0, 1].set_title('x^2')
axes[1, 0].plot(x, x**3);  axes[1, 0].set_title('x^3')
axes[1, 1].plot(x, x**4);  axes[1, 1].set_title('x^4')
fig.suptitle('Powers of x - sharex=True')
fig.tight_layout();


### Saving to PNG

In [ ]:
fig, ax = plt.subplots()
ax.plot(x, x**2)
ax.set_title('Saved')
fig.savefig('/tmp/square.png', dpi=120, bbox_inches='tight')
print('wrote /tmp/square.png');


### Style sheets - change the whole vibe in one line

In [ ]:
with plt.style.context('seaborn-v0_8-whitegrid'):
    fig, ax = plt.subplots()
    ax.plot(x, np.sin(2 * np.pi * x))
    ax.set_title('with seaborn-v0_8-whitegrid');


## 4. Beyond the basics

Ten everyday tools the official Matplotlib user guide hammers — and that you will hit within a week of real work.

### Annotations & text

`ax.text` drops a label at `(x, y)`. `ax.annotate` adds an arrow from text to a target point — great for peaks, outliers, regime changes.

In [ ]:
x = np.linspace(0, 4*np.pi, 200)
fig, ax = plt.subplots()
ax.plot(x, np.sin(x))
ax.annotate('first peak', xy=(np.pi/2, 1.0), xytext=(2.5, 1.3),
            arrowprops=dict(arrowstyle='->', color='#b45309'))
ax.text(4*np.pi - 0.5, -0.9, 'end of window', ha='right', color='#64748b')
plt.show()

### Axis scales — log, symlog, logit

Use `'log'` for orders-of-magnitude data (loss curves, populations). A straight line on log-y means exponential decay.

In [ ]:
epochs = np.arange(1, 101)
loss = 1.0 / epochs + 0.01
fig, ax = plt.subplots()
ax.plot(epochs, loss)
ax.set_yscale('log')
ax.set_xlabel('epoch'); ax.set_ylabel('loss')
plt.show()

### Date / time ticks

Matplotlib understands `datetime` and `numpy.datetime64`. `matplotlib.dates` gives you locators (where ticks go) and formatters (how they render).

In [ ]:
import matplotlib.dates as mdates
import pandas as pd

dates = pd.date_range('2026-01-01', periods=120, freq='D')
y = np.cumsum(np.random.default_rng(0).standard_normal(120))

fig, ax = plt.subplots()
ax.plot(dates, y)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
fig.autofmt_xdate()
plt.show()

### Colorbars

`scatter` / `imshow` / etc. return a *mappable*. Pass it to `fig.colorbar` together with the Axes it should attach to.

In [ ]:
rng = np.random.default_rng(0)
x, y, z = rng.standard_normal((3, 300))

fig, ax = plt.subplots()
sc = ax.scatter(x, y, c=z, cmap='viridis', s=20)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('z value')
plt.show()

### `imshow` — images & heatmaps

Any 2-D array becomes a heatmap. Confusion matrices, attention maps, raw pixels — same call.

- `origin='lower'` puts row 0 at the bottom (math convention).
- `extent=[x0,x1,y0,y1]` maps array indices to data coordinates.
- `aspect='equal'` keeps cells square.

In [ ]:
rng = np.random.default_rng(0)
M = rng.random((8, 8))

fig, ax = plt.subplots()
im = ax.imshow(M, cmap='magma', origin='lower',
               extent=[0, 8, 0, 8], aspect='equal')
fig.colorbar(im, ax=ax)
plt.show()

### Constrained layout — the modern `tight_layout`

`tight_layout()` runs once. `layout='constrained'` runs every draw and adapts automatically — including for colorbars and suptitles. Prefer it for any new multi-subplot figure.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 5), layout='constrained')
for ax in axes.flat:
    ax.plot([0, 1], [0, 1]); ax.set_title('a long title')
fig.suptitle('layout="constrained" handles this automatically')
plt.show()

### `subplot_mosaic` — layouts you can read

Asymmetric layouts used to need `gridspec`. `subplot_mosaic` takes an ASCII map and returns a dict keyed by your labels.

In [ ]:
fig, axd = plt.subplot_mosaic('''
    AAB
    CCB
    ''', figsize=(8, 4), layout='constrained')

axd['A'].plot([0,1,2], [0,1,4]); axd['A'].set_title('A - wide top')
axd['B'].hist(np.random.default_rng(0).standard_normal(200))
axd['B'].set_title('B - full-height right')
axd['C'].plot([0,1,2], [2,1,0]); axd['C'].set_title('C - wide bottom')
plt.show()

### Twin axes — two y-scales, one chart

`ax.twinx()` returns a second Axes sharing the x-axis. Use sparingly (readers can be misled by independent y-scales) but it is the right call for different units.

In [ ]:
x = np.linspace(0, 10, 200)
fig, ax = plt.subplots()
l1, = ax.plot(x, np.sin(x), color='#2563eb', label='signal')
ax.set_ylabel('signal', color='#2563eb')

ax2 = ax.twinx()
l2, = ax2.plot(x, np.exp(x/5), color='#b45309', label='magnitude')
ax2.set_ylabel('magnitude', color='#b45309')

ax.legend(handles=[l1, l2], loc='upper left')
plt.show()

### `rcParams` — defaults you set once

Every default lives in `plt.rcParams`. Set it once and the rest of the session inherits. `plt.style.available` lists built-in style sheets.

In [ ]:
import matplotlib as mpl
old = dict(mpl.rcParams)   # save so we can restore
mpl.rcParams['figure.dpi']     = 110
mpl.rcParams['figure.figsize'] = (6, 3.5)
mpl.rcParams['axes.grid']      = True
mpl.rcParams['lines.linewidth'] = 1.8

fig, ax = plt.subplots()
ax.plot(np.linspace(0, 2*np.pi, 100), np.sin(np.linspace(0, 2*np.pi, 100)))
ax.set_title('rcParams applied globally')
plt.show()

print('first 5 built-in styles:', plt.style.available[:5])
mpl.rcParams.update(old)   # restore

### Legend placement

`loc=` picks a corner; `bbox_to_anchor` places the legend at an arbitrary coordinate. For two legends on one Axes, add the first as an artist before calling `legend` again.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), layout='constrained')
l1, = ax.plot([0,1,2], [0,1,4], label='line A')
l2, = ax.plot([0,1,2], [4,2,1], label='line B')
l3, = ax.plot([0,1,2], [1,2,3], '--', label='reference')

leg1 = ax.legend(handles=[l1, l2], loc='upper left', title='series')
ax.add_artist(leg1)
ax.legend(handles=[l3], loc='center left',
          bbox_to_anchor=(1.02, 0.5), title='lines')
plt.show()

### Recap
* Always start with `fig, ax = plt.subplots()` - explicit beats implicit.
* Pick the right chart for the data: distribution -> hist/box, relation -> scatter, count -> bar.
* Subplots share with `sharex=`/`sharey=`; finish with `fig.tight_layout()`.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

# Exercise 1 - Sine + cosine

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt


**Task 1.** Plot `sin(x)` and `cos(x)` on the same axes for `x` in `[-2 pi, 2 pi]`.

In [ ]:
# your code here


**Task 2.** Add a title, axis labels, a legend, and a grid.

In [ ]:
# your code here


**Task 3.** Use different linestyles for sin (`-`) and cos (`--`).

In [ ]:
# your code here


### Exercise 1 - Solution

# Exercise 1 - Solutions

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
x = np.linspace(-2*np.pi, 2*np.pi, 400)
fig, ax = plt.subplots()
ax.plot(x, np.sin(x), '-',  label='sin')
ax.plot(x, np.cos(x), '--', label='cos')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('sin and cos')
ax.grid(True)
ax.legend();


---

## Exercise 2

# Exercise 2 - Penguin scatter grid

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
%matplotlib inline
import matplotlib.pyplot as plt
from shared.data_utils import load_dataset
penguins = load_dataset('penguins').dropna()


**Task 1.** There are 3 species (`Adelie`, `Chinstrap`, `Gentoo`). Use `plt.subplots(2, 2)` to make a 2x2 grid.

In [ ]:
# your code here


**Task 2.** In each of the first three Axes, draw a scatter of `bill_length_mm` vs `bill_depth_mm` for one species, colored by `sex`.

In [ ]:
# your code here


**Task 3.** Set per-Axes titles to the species name, add `fig.suptitle('Penguins - bill morphology')`, and call `fig.tight_layout()`.

In [ ]:
# your code here


### Exercise 2 - Solution

# Exercise 2 - Solutions

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
%matplotlib inline
import matplotlib.pyplot as plt
from shared.data_utils import load_dataset
penguins = load_dataset('penguins').dropna()

fig, axes = plt.subplots(2, 2, figsize=(9, 7))
species = ['Adelie', 'Chinstrap', 'Gentoo']
for ax, sp in zip(axes.flat, species):
    sub = penguins[penguins['species'] == sp]
    c = (sub['sex'] == 'Female').astype(int)
    ax.scatter(sub['bill_length_mm'], sub['bill_depth_mm'],
               c=c, cmap='coolwarm', alpha=0.7)
    ax.set_title(sp)
    ax.set_xlabel('bill length (mm)')
    ax.set_ylabel('bill depth (mm)')
axes[1, 1].axis('off')
fig.suptitle('Penguins - bill morphology')
fig.tight_layout();


---

## Exercise 3

# Exercise 3 - Histogram with reference line

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
%matplotlib inline
import matplotlib.pyplot as plt
from shared.data_utils import load_dataset
tips = load_dataset('tips')


**Task 1.** Plot a histogram of `tips['total_bill']` with 30 bins.

In [ ]:
# your code here


**Task 2.** Overlay a vertical line at the mean using `ax.axvline`; label it in a legend.

In [ ]:
# your code here


**Task 3.** Add axis labels and a title; save the figure to `/tmp/tips_hist.png`.

In [ ]:
# your code here


### Exercise 3 - Solution

# Exercise 3 - Solutions

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
%matplotlib inline
import matplotlib.pyplot as plt
from shared.data_utils import load_dataset
tips = load_dataset('tips')

fig, ax = plt.subplots()
ax.hist(tips['total_bill'], bins=30, rwidth=0.9)
mu = tips['total_bill'].mean()
ax.axvline(mu, color='red', linestyle='--', label=f'mean = {mu:.2f}')
ax.set_xlabel('total_bill ($)')
ax.set_ylabel('count')
ax.set_title('Distribution of total_bill in tips')
ax.legend()
fig.savefig('/tmp/tips_hist.png', dpi=120, bbox_inches='tight')
print('saved /tmp/tips_hist.png');
